# PKG — Insurer / Subrogation Ego-Network Explorer
**Snapshot:** 2025-11 &nbsp;|&nbsp; **Focus:** insurer-role nodes and their first-order payment neighborhood

This notebook:
1. Loads the 2025-11 monthly snapshot
2. Splits `source_naics` / `dest_naics` (`code|description`) and derives 2/3/4/5/6-digit NAICS levels
3. Builds the full graph in **NetworkIt** (fast enough at 3-5M edges; NetworkX's pure-Python
   dict-of-dict backend gets uncomfortably slow well before this scale for anything beyond
   trivial iteration)
4. Identifies insurer / subrogation-relevant seed nodes by NAICS + name regex
5. Extracts the first-order ego network around those seeds
6. Hands the (now small) ego subgraph to **NetworkX** for flexible, attribute-rich analysis
7. Looks at payer/payee structure, NAICS composition, and insurer↔insurer reciprocity
   (the structural fingerprint of subrogation reimbursement)

In [ ]:
import pandas as pd
import numpy as np
import re
import networkit as nk
import networkx as nx

pd.set_option("display.max_colwidth", 60)

## 1. Config & load

Point `SNAPSHOT_PATH` at the 2025-11 CSV (or swap this cell for your existing Impala/Hive
pull if you're reading from the lab tables instead of a local file — same column set).

In [ ]:
SNAPSHOT_PATH = "2025-11.csv"  # <-- update to your actual path / Impala query

cols = ["source", "source_name", "source_naics", "amount", "volume",
        "dest", "dest_name", "dest_naics"]

df = pd.read_csv(SNAPSHOT_PATH, usecols=cols, dtype={"source": "string", "dest": "string"})
print(f"rows: {len(df):,}")
df.head(3)

## 2. NAICS: split `code|description`, derive 2/3/4/5/6-digit levels

Handles the known data-quality wrinkles (`-1|UNKNOWN`, `******`, missing separator) by
only truncating codes that are purely numeric and 2-6 digits long — anything else becomes
`NA` at every level rather than silently producing a garbage prefix.

In [ ]:
def split_naics(series: pd.Series):
    parts = series.astype("string").str.split("|", n=1, expand=True)
    code_ = parts[0].str.strip()
    desc_ = parts[1].str.strip() if parts.shape[1] > 1 else pd.Series(pd.NA, index=series.index, dtype="string")
    return code_, desc_

def naics_levels(code_series: pd.Series, prefix: str) -> pd.DataFrame:
    valid = code_series.where(code_series.str.fullmatch(r"\d{2,6}"), other=pd.NA)
    out = {}
    for n in (2, 3, 4, 5, 6):
        out[f"{prefix}_naics{n}"] = valid.where(valid.str.len() >= n, other=pd.NA).str.slice(0, n)
    return pd.DataFrame(out, index=code_series.index)

df["source_naics_code"], df["source_naics_desc"] = split_naics(df["source_naics"])
df["dest_naics_code"],   df["dest_naics_desc"]   = split_naics(df["dest_naics"])

df = pd.concat([
    df,
    naics_levels(df["source_naics_code"], "source"),
    naics_levels(df["dest_naics_code"], "dest"),
], axis=1)

df[["source_naics_code", "source_naics2", "source_naics3",
    "source_naics4", "source_naics5", "source_naics6"]].drop_duplicates().head(8)

## 3. Node table

Each node can appear as both a `source` and a `dest` across many rows. We take the
first non-null name/NAICS observed per node id — `.groupby().first()` is vectorized
(unlike a custom per-group callable, which stalls badly at this row count, per the
`groupby.nlargest` lesson from the monthly metrics pipeline).

In [ ]:
def node_frame(frame: pd.DataFrame, side: str) -> pd.DataFrame:
    rename = {
        side: "node_id", f"{side}_name": "name",
        f"{side}_naics_code": "naics_code", f"{side}_naics_desc": "naics_desc",
        f"{side}_naics2": "naics2", f"{side}_naics3": "naics3",
        f"{side}_naics4": "naics4", f"{side}_naics5": "naics5", f"{side}_naics6": "naics6",
    }
    return frame[list(rename.keys())].rename(columns=rename)

nodes_long = pd.concat([node_frame(df, "source"), node_frame(df, "dest")], ignore_index=True)
nodes = nodes_long.groupby("node_id", sort=False).first().reset_index()
print(f"unique nodes: {len(nodes):,}")
nodes.head(3)

## 4. Edge table — amount & volume as weights

`amount` is summed as the raw accounting weight; `log_amount` (`log1p`) is kept alongside
for anything spectral/iterative later, matching the convention used in the monthly metrics
pipeline (raw for flow/accounting, log1p for spectral/iterative measures).

In [ ]:
edges = (df.groupby(["source", "dest"], as_index=False)
            .agg(amount=("amount", "sum"), volume=("volume", "sum")))
edges["log_amount"] = np.log1p(edges["amount"])
print(f"unique source-dest pairs: {len(edges):,}")
edges.head(3)

## 5. Build the full graph — NetworkIt

At 3-5M edges, NetworkIt (Cython/C++ backend, purpose-built for million-node-scale graphs)
is the right choice for the *full* snapshot graph. NetworkX is kept for the much smaller
induced ego-subgraph later, where its convenience and algorithm library are the bigger win
and scale is no longer a concern — same split already reflected in the target architecture
(Neo4j/Cypher result → NetworkX, not the raw multi-million-edge graph).

NetworkIt needs contiguous integer node ids, so we map `node_id` → index once and reuse
the mapping everywhere below.

In [ ]:
node_ids = nodes["node_id"].to_numpy()
id_to_idx = {nid: i for i, nid in enumerate(node_ids)}
n = len(node_ids)

G = nk.Graph(n, weighted=True, directed=True)

src_idx = edges["source"].map(id_to_idx).to_numpy()
dst_idx = edges["dest"].map(id_to_idx).to_numpy()
w = edges["log_amount"].to_numpy()

for s, d, wt in zip(src_idx, dst_idx, w):
    G.addEdge(s, d, wt)

print(f"networkit graph — nodes: {G.numberOfNodes():,}  edges: {G.numberOfEdges():,}")

## 6. Seed nodes — NAICS + name regex

Core insurer NAICS (P&C carriers, reinsurance) plus an adjacent bucket (claims adjusting /
TPA, since they often sit in the settlement path) plus a name-regex fallback for anything
missing/misclassified NAICS. Adjust the code sets and pattern as you see what the real
data surfaces — NAICS unknowns (`-1`, `******`) are common per the roadmap's known data
quality issues.

In [ ]:
CORE_INSURER_NAICS6 = {"524126", "524130"}   # Direct P&C carriers, Reinsurance carriers
ADJACENT_NAICS6 = {"524291", "524292"}       # Claims adjusting, TPA of insurance/pension funds

NAME_PATTERN = r"\b(?:insur\w*|assuranc\w*|indemnit\w*|casualty|mutual|reinsur\w*|underwrit\w*)\b"

is_core = nodes["naics6"].isin(CORE_INSURER_NAICS6)
is_adj = nodes["naics6"].isin(ADJACENT_NAICS6)
is_name = nodes["name"].fillna("").str.contains(NAME_PATTERN, regex=True, case=False)

nodes["seed_role"] = np.select(
    [is_core, is_adj & ~is_core, is_name & ~is_core & ~is_adj],
    ["core_insurer", "claims_adjacent", "name_match_only"],
    default=None,
)

seed_nodes = nodes.loc[nodes["seed_role"].notna()].copy()
seed_nodes["seed_role"].value_counts()

### 6a. Hub sanity check before expanding

If any seed node is itself a mega-hub (per the roadmap's ~100K-degree hub nodes — PNC
internal accounts, other-bank accounts, processors, payroll), expanding its neighborhood
will flood the ego set with irrelevant nodes. Flag anything with unusually high degree
before proceeding — you may want to exclude it from expansion (still analyze it as a seed,
just don't pull in all its neighbors).

In [ ]:
seed_idx_map = {nid: id_to_idx[nid] for nid in seed_nodes["node_id"] if nid in id_to_idx}

seed_degree = pd.DataFrame({
    "node_id": list(seed_idx_map.keys()),
    "out_deg": [G.degreeOut(i) for i in seed_idx_map.values()],
    "in_deg":  [G.degreeIn(i) for i in seed_idx_map.values()],
})
seed_degree["total_deg"] = seed_degree["out_deg"] + seed_degree["in_deg"]

HUB_DEGREE_THRESHOLD = 5000  # tune against your hub exclusion registry
hub_like = seed_degree[seed_degree["total_deg"] > HUB_DEGREE_THRESHOLD]
print(f"seed nodes flagged as hub-like (degree > {HUB_DEGREE_THRESHOLD}): {len(hub_like)}")
hub_like.sort_values('total_deg', ascending=False).head(10)

## 7. First-order ego network

Seed nodes + their direct in/out neighbors, then induce all edges among that combined
set (standard ego-graph definition — not just tree edges from the seeds). Any node
flagged as hub-like above is excluded from the *expansion* step but stays in the seed
list itself.

In [ ]:
expand_from = set(seed_degree.loc[seed_degree["total_deg"] <= HUB_DEGREE_THRESHOLD, "node_id"].map(id_to_idx))
seed_idx_all = set(seed_idx_map.values())

ego_idx = set(seed_idx_all)
for i in expand_from:
    ego_idx.update(G.iterNeighbors(i))
    ego_idx.update(G.iterInNeighbors(i))

ego_ids = {node_ids[i] for i in ego_idx}
print(f"seed nodes: {len(seed_idx_all):,}  |  ego node set (incl. neighbors): {len(ego_ids):,}")

ego_edges = edges[edges["source"].isin(ego_ids) & edges["dest"].isin(ego_ids)].copy()
print(f"induced edges: {len(ego_edges):,}")

## 8. Hand off to NetworkX

The ego subgraph is now small enough that NetworkX's convenience (rich attributes,
algorithm library, easy `pyvis`/Plotly export) outweighs any performance concern.

In [ ]:
node_attr_cols = ["node_id", "name", "naics_code", "naics_desc", "naics2", "naics3",
                   "naics4", "naics5", "naics6", "seed_role"]
ego_node_attrs = nodes.loc[nodes["node_id"].isin(ego_ids), node_attr_cols].set_index("node_id")

Gx = nx.DiGraph()
for nid, row in ego_node_attrs.iterrows():
    Gx.add_node(nid, **row.to_dict())
for _, r in ego_edges.iterrows():
    Gx.add_edge(r["source"], r["dest"], amount=r["amount"], volume=r["volume"], log_amount=r["log_amount"])

print(f"networkx ego graph — nodes: {Gx.number_of_nodes():,}  edges: {Gx.number_of_edges():,}")

## 9. Payer / payee analysis

Top counterparties paying *into* seed insurer nodes, and top counterparties insurers pay
*out* to — the basic payer/payee split before digging into anything structural.

In [ ]:
seed_id_set = set(seed_nodes["node_id"])

pay_in = ego_edges[ego_edges["dest"].isin(seed_id_set)]
pay_out = ego_edges[ego_edges["source"].isin(seed_id_set)]

print("Top payers -> insurer nodes (by amount):")
display(pay_in.nlargest(10, "amount")[["source", "dest", "amount", "volume"]])

print("\nTop insurer -> payee flows (by amount):")
display(pay_out.nlargest(10, "amount")[["source", "dest", "amount", "volume"]])

## 10. NAICS composition of the insurer neighborhood

What industries actually sit around insurer nodes — e.g. a heavy showing from repair
(NAICS `8112xx`) or medical (`621xxx`) sectors on the payee side, vs. financial-sector
(`52xxxx`) concentration on the payer side, is exactly the kind of structural signal
that doesn't show up in a flat SQL groupby.

In [ ]:
naics_lookup = nodes.set_index("node_id")["naics2"]

pay_in_sector = (pay_in.assign(counterparty_naics2=pay_in["source"].map(naics_lookup))
                  .groupby("counterparty_naics2")["amount"].sum()
                  .sort_values(ascending=False).head(10))

pay_out_sector = (pay_out.assign(counterparty_naics2=pay_out["dest"].map(naics_lookup))
                   .groupby("counterparty_naics2")["amount"].sum()
                   .sort_values(ascending=False).head(10))

print("Amount into insurers by counterparty NAICS2:")
display(pay_in_sector)
print("\nAmount out of insurers by counterparty NAICS2:")
display(pay_out_sector)

## 11. Insurer ↔ insurer reciprocity (subrogation fingerprint)

The most distinctive structural signature of subrogation: two carriers paying each other
in *both* directions, which is rare for ordinary commercial relationships (premium flow,
agency commissions, treaty settlement tend to be directionally consistent). Flags
candidate pairs and the amount asymmetry between the two directions.

In [ ]:
insurer_ids = set(seed_nodes.loc[seed_nodes["seed_role"] == "core_insurer", "node_id"])
ii_edges = ego_edges[ego_edges["source"].isin(insurer_ids) & ego_edges["dest"].isin(insurer_ids)]

pair_amounts = ii_edges.set_index(["source", "dest"])["amount"]
pair_set = set(pair_amounts.index)
reciprocal_pairs = sorted({tuple(sorted((a, b))) for (a, b) in pair_set if (b, a) in pair_set})

recip_rows = []
for a, b in reciprocal_pairs:
    amt_ab = pair_amounts.get((a, b), 0.0)
    amt_ba = pair_amounts.get((b, a), 0.0)
    recip_rows.append({
        "insurer_a": a, "insurer_b": b,
        "a_to_b_amount": amt_ab, "b_to_a_amount": amt_ba,
        "net_flow": amt_ab - amt_ba,
        "total_flow": amt_ab + amt_ba,
    })

reciprocity_df = pd.DataFrame(recip_rows).sort_values("total_flow", ascending=False)
print(f"insurer<->insurer edges: {len(ii_edges):,}  |  reciprocal pairs: {len(reciprocity_df):,}")
reciprocity_df.head(15)

## 12. Next steps

- If reciprocal pairs turn up, cross-reference amount bands against typical claim size
  (small/frequent = claims subrogation, large/infrequent = more likely treaty reinsurance)
- Push the flagged insurer nodes + ego set into the existing Motif Finder / Enhanced Graph
  Explorer for a visual pass (PyViz) rather than re-building visualization here
- If this looks promising, this slots into the roadmap next to **#9 Corporate Client
  Relationship Scoring** and **#3 Funnel/Motif Detector** as an insurance-vertical variant,
  reusing the same infrastructure with an insurer node-role filter